# Distribution Gallery

Chapter 13: Probability and Random Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Interactive PMFs and PDFs for Uniform, Bernoulli, Binomial, Categorical, Gaussian, Beta, Dirichlet, and GMM.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-13/distribution-gallery/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch
import torch.distributions as D

torch.manual_seed(0)

# The demo's default parameters for each family
gallery = {
    "Uniform(0, 2)": D.Uniform(0.0, 2.0),
    "Bernoulli(0.4)": D.Bernoulli(probs=torch.tensor(0.4)),
    "Binomial(10, 0.35)": D.Binomial(10, probs=torch.tensor(0.35, dtype=torch.float64)),
    "Categorical(0.5, 0.3, 0.2)": D.Categorical(probs=torch.tensor([0.5, 0.3, 0.2])),
    "Gaussian(0, 1)": D.Normal(0.0, 1.0),
    "Beta(2, 5)": D.Beta(torch.tensor(2.0, dtype=torch.float64), torch.tensor(5.0, dtype=torch.float64)),
    "Dirichlet(2, 2, 2)": D.Dirichlet(torch.tensor([2.0, 2.0, 2.0])),
}

# ── Each family: theoretical mean/variance vs 200,000 samples ───────────────
print(f"{'distribution':28s} {'mean':>18s}   {'sample mean':>18s}   {'variance':>18s}")
for name, dist in gallery.items():
    x = dist.sample((200000,)).double()
    if name.startswith("Categorical"):
        mean = (torch.arange(3) * dist.probs).sum()
        var = ((torch.arange(3) - mean) ** 2 * dist.probs).sum()
        print(f"{name:28s} {mean.item():18.4f}   {x.mean().item():18.4f}   {var.item():18.4f}")
        continue
    mean, var = dist.mean, dist.variance
    fmt = lambda t: str([round(v, 4) for v in t.tolist()]) if t.dim() else f"{t.item():.4f}"
    print(f"{name:28s} {fmt(mean):>18s}   {fmt(x.mean(0)):>18s}   {fmt(var):>18s}")

# ── Discrete PMFs sum to 1; densities integrate to 1 ───────────────────────
binom = gallery["Binomial(10, 0.35)"]
pmf = binom.log_prob(torch.arange(11, dtype=torch.float64)).exp()
print("\nBinomial PMF sums to", pmf.sum().item(), "| mode k =", pmf.argmax().item())
grid = torch.linspace(1e-6, 1 - 1e-6, 200001, dtype=torch.float64)
beta_pdf = gallery["Beta(2, 5)"].log_prob(grid).exp()
print("Beta(2,5) density integrates to", torch.trapezoid(beta_pdf, grid).item())

# ── Gaussian mixture: 0.4 N(-1.5, 0.6^2) + 0.6 N(1.8, 0.5^2) ────────────────
gmm = D.MixtureSameFamily(D.Categorical(torch.tensor([0.4, 0.6])),
                          D.Normal(torch.tensor([-1.5, 1.8]), torch.tensor([0.6, 0.5])))
s = gmm.sample((200000,))
print("\nGMM mean =", gmm.mean.item(), " sample mean =", s.mean().item())
print("fraction of samples below 0 =", (s < 0).float().mean().item(), "(about the first weight, 0.4)")

distribution                               mean          sample mean             variance
Uniform(0, 2)                            1.0000               1.0016               0.3333
Bernoulli(0.4)                           0.4000               0.3999               0.2400
Binomial(10, 0.35)                       3.5000               3.4950               2.2750
Categorical(0.5, 0.3, 0.2)               0.7000               0.7010               0.6100
Gaussian(0, 1)                           0.0000              -0.0011               1.0000
Beta(2, 5)                               0.2857               0.2866               0.0255
Dirichlet(2, 2, 2)           [0.3333, 0.3333, 0.3333]   [0.3335, 0.3337, 0.3328]   [0.0317, 0.0317, 0.0317]

Binomial PMF sums to 0.9999999999999998 | mode k = 3
Beta(2,5) density integrates to 0.9999999999225009

GMM mean = 0.48000001907348633  sample mean = 0.47658008337020874
fraction of samples below 0 = 0.3986549973487854 (about the first weight, 0.4)
